# Evaluación de Omnipose sobre el corpus de mitocondrias

**Barrido de hiperparámetros de inferencia** de modelos Omnipose pre-entrenados,
evaluados *zero-shot* contra el ground truth anotado a nivel de instancia.

> ⚠️ Esto **no es fine-tuning**: no se actualizan los pesos. Es un barrido de
> parámetros de `model.eval()` de Omnipose (`mask_threshold`, `flow_threshold`),
> con reconstrucción Omni activada. Son cosas distintas en la tesis (CLAUDE.md).

Se evalúan **varios modelos pre-entrenados y se comparan** (`bact_fluor_omni`,
`cyto2_omni`, `worm_bact_omni`).

## Metodología (según CLAUDE.md, idéntica a la de Cellpose)
- Matching uno-a-uno por **algoritmo húngaro** sobre `1 - IoU` (no *greedy*).
- Métricas a nivel instancia con τ **parametrizable**; barrido
  τ ∈ {0.5, …, 0.9}, con τ = 0.5 como referencia.
- Se elige **una única configuración global por modelo**, la que maximiza la
  métrica **promediada sobre todas las imágenes** — nunca un óptimo por-imagen
  (eso sería un oráculo que usa el GT de cada imagen).
- **Una sola** implementación de `iou_matrix`, reutilizada por métricas y
  visualización.


## Diferencias respecto del notebook de Cellpose

El bloque de **inferencia** cambia (el de métricas/evaluación es idéntico):

| Aspecto | Cellpose-SAM | Omnipose |
|---|---|---|
| Librería | `cellpose` | `omnipose` + `cellpose_omni` |
| Modelo | `CellposeModel(gpu=True)` (SAM) | `CellposeModel(gpu=True, model_type=<nombre>)` |
| Umbral de segmentación | `cellprob_threshold` | **`mask_threshold`** (≈ -5..5) + `omni=True` |
| Extras de eval | — | `cluster` (DBSCAN), `resample`, `niter=None` (auto) |

**Errores del notebook previo (`Fine_tunning_omnipose.ipynb`) que se corrigen:**
`train_with_parameters` roto; fórmulas de precision/f1 mal; selección de
hiperparámetros por imagen (oráculo); `iou_matrix` duplicada (la de la viz asumía
fondo en el primer ID con `[1:]`); τ hardcodeado; faltaban `error_rate` y
`ap_cellpose`; y la grilla barría `cellprob_threshold` (parámetro de Cellpose) en
vez de `mask_threshold` (el de Omnipose).

> El módulo de métricas se repite tal cual el del notebook de Cellpose porque
> Colab necesita el notebook self-contained. En la tesis, idealmente vive en un
> módulo compartido único.


## Instalación (siguiendo la guía oficial de Omnipose)

La guía recomienda instalar el **release de PyPI** con `pip install omnipose`, y su
*Warning* pide **limpiar** cualquier `cellpose_omni` previo para evitar conflictos
de versión (justo el problema que tuvimos). Sobre el Python de Colab agregamos el
pin **`ncolor<2`**: las versiones 2.x de `ncolor` quitaron `unique_nonzero`, que la
Omnipose de PyPI todavía importa — esa era la causa del `ImportError`.

> ⚠️ **Después de correr la celda de instalación, Runtime → Restart session** y
> recién ahí seguí con `drive.mount` e imports. Sin reiniciar, queda cargada en
> memoria la `ncolor` vieja y el error persiste.


### Solución para error de `lxml` y compatibilidad de Python

El error `subprocess-exited-with-error` durante la instalación de `lxml` y el `ModuleNotFoundError` para `cellpose_omni` son probablemente causados por la incompatibilidad de la versión de Python. Tu entorno actual usa Python 3.13, pero Omnipose está confirmado para funcionar con Python 3.10 o anterior. Esto hace que `pip` no encuentre wheels precompiladas para `lxml` y falle al intentar construirlo desde el código fuente.

Instalaremos `libxml2-dev`, `libxslt1-dev` y `build-essential` (que provee herramientas de compilación) para intentar resolver los problemas de construcción. Sin embargo, la versión de Python sigue siendo el problema principal. Si esto no funciona, la solución más confiable sería usar un entorno de Colab con una versión de Python compatible.

In [ ]:
# Limpieza recomendada por la guía (Warning) + restos de instalaciones previas.
!pip uninstall -y omnipose cellpose_omni ncolor
!pip cache remove cellpose_omni

# Release de PyPI de Omnipose + pin de ncolor<2 (necesario en el Python de Colab).
!pip install --no-cache-dir omnipose "ncolor<2"

print("\n>>> Ahora: Runtime -> Restart session, y seguí con la celda de drive.mount <<<")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

from cellpose_omni import io, models

print("Modelos Omnipose disponibles:")
print(models.MODEL_NAMES)

In [ ]:
# --- Rutas (Colab / Google Drive) ---
# Carpeta con los 15 frames del conjunto de evaluación (cada .tif con su _seg.npy al lado).
IMAGES_DIR = "/content/drive/MyDrive/Tesis Santiago Feliu/Notebooks/Cellpose/Images_eval15"
OUTPUT_DIR = "/content/drive/MyDrive/Tesis Santiago Feliu/Notebooks/Omnipose/predicted_masks_eval15"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Modelos Omnipose pre-entrenados a evaluar y comparar ---
MODELS = ["bact_fluor_omni", "cyto2_omni"]

# --- Grilla de hiperparámetros de INFERENCIA de Omnipose (no fine-tuning) ---
PARAM_GRID_OMNI = {
    "mask_threshold": [-4, -2, -1, 0, 1],  # umbral de segmentación de Omnipose (~ -5..5)
    "flow_threshold": [0.0, 0.4],          # 0 desactiva el filtrado por error de flujo
}

# Parámetros fijos de model.eval (reconstrucción Omni + clustering).
EVAL_PARAMS = {
    "omni": True,        # reconstrucción de máscaras de Omnipose
    "cluster": True,     # clustering DBSCAN
    "niter": None,       # Omnipose calcula el nº de iteraciones automáticamente
    "channels": None,    # imágenes en escala de grises, sin mapeo de canales
    "resample": True,
    "affinity_seg": False,
    "verbose": False,
    "tile": False,
}

# --- Umbrales de IoU para el matching a nivel instancia ---
TAUS = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
TAU_REF = 0.5  # valor de referencia

# --- Frames de referencia para la visualización cualitativa (conjunto de evaluación) ---
FRAMES_REFERENCIA = {
    "P6T10":   "ImageFV10006.50006_C002T010",
    "P10T56":  "ImageFV10006.50010_C002T056",
    "P10T93":  "ImageFV10006.50010_C002T093",
    "P10T134": "ImageFV10006.50010_C002T134",
    "P24T23":  "ImageFV10006.50024_C002T023",
    "P24T38":  "ImageFV10006.50024_C002T038",
    "P24T61":  "ImageFV10006.50024_C002T061",
    "P24T186": "ImageFV10006.50024_C002T186",
    "P25T20":  "ImageFV10006.50025_C002T020",
    "P25T34":  "ImageFV10006.50025_C002T034",
    "P25T111": "ImageFV10006.50025_C002T111",
    "P7T50":   "ImageFV10006.50007_C002T050",
    "P7T164":  "ImageFV10006.50007_C002T164",
    "P8T47":   "ImageFV10006.50008_C002T047",
    "P8T121":  "ImageFV10006.50008_C002T121",
}


In [ ]:
def get_ground_truth(image_filepath: str) -> dict:
    """Carga el _seg.npy de Cellpose asociado a una imagen .tif."""
    return np.load(image_filepath[:-4] + "_seg.npy", allow_pickle=True).item()


def get_tif_files(directory: str) -> dict:
    """Devuelve {archivo.tif: ruta_completa} para todos los .tif del directorio."""
    return {
        f: os.path.join(directory, f)
        for f in sorted(os.listdir(directory))
        if f.endswith(".tif")
    }


def cargar_imagen(image_filepath: str) -> np.ndarray:
    """Lee un .tif y lo deja en escala de grises 2D (Omnipose normaliza internamente)."""
    img = io.imread(image_filepath)
    if img.ndim == 3:  # si viniera con canal extra, promediar a grayscale
        img = img.mean(axis=-1)
    return img

## Métricas a nivel instancia (idénticas al notebook de Cellpose)

Entrada: dos máscaras de etiquetas 2D (`0` = fondo, `1..K` = IDs de instancia).

1. **Matriz IoU** `M × N`, excluyendo el fondo por valor (`!= 0`).
2. **Matching húngaro** sobre `1 - IoU`; se descartan pares con `IoU < τ`.
3. **Conteos:** `TP` = pares que sobreviven; `FN = M − TP`; `FP = N − TP`.
4. **Métricas:**

```
error_rate  = (FP + FN) / (TP + FN)    # denom = nº de instancias GT; no acotado por 1
ap_cellpose = TP / (TP + FN + FP)      # Jaccard de instancias (convención Cellpose/Omnipose)
precision   = TP / (TP + FP)
recall      = TP / (TP + FN)
f1          = 2·P·R / (P + R)
```


In [ ]:
def iou_matrix(gt_lab: np.ndarray, pr_lab: np.ndarray):
    """
    Matriz de IoU entre instancias de dos máscaras de etiquetas 2D.

    Ambas máscaras: 0 = fondo, 1..K = IDs de instancia. El fondo se filtra por
    valor (`!= 0`), NO se asume que sea el primer ID. np.unique sólo devuelve
    etiquetas presentes, por lo que las instancias de área 0 no aparecen.

    Retorna (gt_ids, pr_ids, iou), con iou de forma (len(gt_ids), len(pr_ids)).
    """
    gt_ids = np.unique(gt_lab)
    gt_ids = gt_ids[gt_ids != 0]
    pr_ids = np.unique(pr_lab)
    pr_ids = pr_ids[pr_ids != 0]

    iou = np.zeros((len(gt_ids), len(pr_ids)), dtype=float)
    if len(gt_ids) == 0 or len(pr_ids) == 0:
        return gt_ids, pr_ids, iou

    gt_areas = {i: int((gt_lab == i).sum()) for i in gt_ids}
    pr_areas = {j: int((pr_lab == j).sum()) for j in pr_ids}
    for gi, i in enumerate(gt_ids):
        mask_i = (gt_lab == i)
        for pj, j in enumerate(pr_ids):
            inter = int(np.logical_and(mask_i, pr_lab == j).sum())
            if inter > 0:
                iou[gi, pj] = inter / (gt_areas[i] + pr_areas[j] - inter)
    return gt_ids, pr_ids, iou


def match_instances(gt_lab: np.ndarray, pr_lab: np.ndarray, tau: float):
    """
    Matching uno-a-uno GT<->Pred por algoritmo húngaro sobre 1 - IoU.
    Descarta los pares asignados con IoU < tau. Reutiliza iou_matrix.
    """
    gt_ids, pr_ids, iou = iou_matrix(gt_lab, pr_lab)

    matched_gt_ids, matched_pr_ids = [], []
    if iou.size > 0:
        gi, pj = linear_sum_assignment(1.0 - iou)
        for g, p in zip(gi, pj):
            if iou[g, p] >= tau:
                matched_gt_ids.append(gt_ids[g])
                matched_pr_ids.append(pr_ids[p])

    tp = len(matched_pr_ids)
    fp = len(pr_ids) - tp
    fn = len(gt_ids) - tp
    return {
        "tp": tp, "fp": fp, "fn": fn,
        "tp_pr_ids": set(matched_pr_ids),
        "fp_pr_ids": set(pr_ids) - set(matched_pr_ids),
        "fn_gt_ids": set(gt_ids) - set(matched_gt_ids),
        "n_gt": len(gt_ids), "n_pr": len(pr_ids),
    }


def instance_metrics(gt_lab: np.ndarray, pr_lab: np.ndarray, tau: float = TAU_REF):
    """
    Métricas a nivel instancia para un τ dado. Si M = 0 (sin GT), retorna None
    para excluir la imagen del promedio (denominador cero).
    """
    m = match_instances(gt_lab, pr_lab, tau)
    tp, fp, fn = m["tp"], m["fp"], m["fn"]
    if (tp + fn) == 0:
        return None

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    error_rate = (fp + fn) / (tp + fn)
    ap_cellpose = tp / (tp + fn + fp) if (tp + fn + fp) > 0 else 0.0
    return {
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
        "error_rate": error_rate, "ap_cellpose": ap_cellpose,
    }

## Métricas pixel a pixel (como Ilastik)

Los modelos de instancias también se evalúan a nivel **píxel**: binarizando la
máscara (`label > 0` → primer plano) y aplicando la misma matriz de confusión
foreground/fondo que usa Ilastik. **No** depende de τ; se calcula una vez por
predicción.


In [ ]:
from matplotlib.colors import ListedColormap

_PX_CMAP = ListedColormap(["black", "white", "green", "red"])  # TN, TP, FN, FP


def pixel_metrics(gt_lab: np.ndarray, pr_lab: np.ndarray):
    """
    Métricas pixel a pixel (foreground vs fondo), misma convención que Ilastik
    (calculate_segmentation_metrics). Binariza label > 0. Columnas con prefijo px_.
    """
    y_true = gt_lab > 0
    y_pred = pr_lab > 0
    tp = int(np.sum(y_true & y_pred))
    tn = int(np.sum(~y_true & ~y_pred))
    fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    return {
        "px_tp": tp, "px_tn": tn, "px_fp": fp, "px_fn": fn,
        "px_precision": precision, "px_recall": recall, "px_f1": f1,
        "px_iou": iou, "px_specificity": specificity, "px_accuracy": accuracy,
        "px_dice": f1,
    }


def pixel_superposition(gt_lab: np.ndarray, pr_lab: np.ndarray):
    """Mapa de superposición pixel a pixel (estilo Ilastik):
    0 = fondo (TN, negro), 1 = TP (blanco), 2 = FN (verde), 3 = FP (rojo)."""
    y_true = gt_lab > 0
    y_pred = pr_lab > 0
    sup = np.zeros(gt_lab.shape, dtype=np.uint8)
    sup[y_true & y_pred] = 1
    sup[y_true & ~y_pred] = 2
    sup[~y_true & y_pred] = 3
    return sup

## Barrido de la grilla sobre TODOS los modelos e imágenes

Por cada **modelo** × imagen × configuración `(mask_threshold, flow_threshold)` se
infiere una vez. Se calculan las métricas de **instancia** por cada τ y las de
**píxel** una sola vez por config. **No** se elige el mejor parámetro por imagen
(oráculo).


In [ ]:
def evaluar_grilla_omni(model_names, param_grid, images, taus=TAUS, eval_params=EVAL_PARAMS):
    """
    Corre cada modelo Omnipose sobre todas las imágenes y toda la grilla. Devuelve
    DOS DataFrames:
      - instancia: una fila por (modelo, imagen, config, τ)
      - píxel:     una fila por (modelo, imagen, config)  [no depende de τ]
    """
    combos = list(itertools.product(
        param_grid["mask_threshold"],
        param_grid["flow_threshold"],
    ))

    filas_inst, filas_px = [], []
    for model_name in model_names:
        print(f"\n### Modelo: {model_name}")
        model = models.CellposeModel(gpu=True, model_type=model_name)

        for image_name, image_filepath in images.items():
            gt_lab = get_ground_truth(image_filepath)["masks"]
            img = cargar_imagen(image_filepath)
            print(f"  {image_name}  ({len(combos)} configs x {len(taus)} τ)")

            for mask_thr, flow_thr in combos:
                pr_lab, _, _ = model.eval(
                    img, mask_threshold=mask_thr, flow_threshold=flow_thr, **eval_params
                )
                cfg_cols = {
                    "model": model_name,
                    "image_name": image_name,
                    "mask_threshold": mask_thr,
                    "flow_threshold": flow_thr,
                }
                # Píxel: una sola vez por config (no depende de τ)
                filas_px.append({**cfg_cols, **pixel_metrics(gt_lab, pr_lab)})
                # Instancia: por cada τ
                for tau in taus:
                    met = instance_metrics(gt_lab, pr_lab, tau=tau)
                    if met is None:  # imagen sin GT -> excluida
                        continue
                    filas_inst.append({**cfg_cols, "tau": tau, **met})
    return pd.DataFrame(filas_inst), pd.DataFrame(filas_px)

In [ ]:
resultados, resultados_px = evaluar_grilla_omni(MODELS, PARAM_GRID_OMNI, get_tif_files(IMAGES_DIR))
resultados.to_csv(f"{OUTPUT_DIR}/resultados_por_imagen.csv", index=False)
resultados_px.to_csv(f"{OUTPUT_DIR}/resultados_pixel_por_imagen.csv", index=False)
print("instancia:", resultados.shape, "| pixel:", resultados_px.shape)
resultados_px.head()

## Selección de la configuración global por modelo

Para cada modelo, a τ = 0.5, se promedia cada métrica sobre las imágenes por
configuración y se elige la que maximiza el promedio. **Una** config por modelo,
no una por imagen.


In [ ]:
def seleccionar_config_por_modelo(resultados, tau=TAU_REF, metrica="f1"):
    """
    Para cada modelo, promedia la métrica sobre imágenes por configuración (a un τ
    fijo) y devuelve: (a) la mejor config de cada modelo, (b) la tabla completa de
    promedios por (modelo, config).
    """
    cols_cfg = ["model", "mask_threshold", "flow_threshold"]
    cols_met = ["precision", "recall", "f1", "error_rate", "ap_cellpose"]
    agg = (
        resultados[resultados["tau"] == tau]
        .groupby(cols_cfg)[cols_met]
        .mean()
        .reset_index()
    )
    mejores = (
        agg.sort_values(metrica, ascending=False)
        .groupby("model", as_index=False)
        .first()
    )
    return mejores, agg


mejores_por_modelo, ranking_configs = seleccionar_config_por_modelo(resultados, tau=TAU_REF, metrica="f1")
ranking_configs.to_csv(f"{OUTPUT_DIR}/config_ranking_por_modelo.csv", index=False)
print(f"Mejor configuración por modelo (τ={TAU_REF}, por f1 promedio sobre imágenes):")
mejores_por_modelo

## Guardado de las predicciones de Omnipose para las 5 mejores configuraciones

A continuación, se identificarán las 5 mejores configuraciones (modelo y parámetros de inferencia) basándose en la métrica `ap_cellpose` promediada sobre todas las imágenes. Para cada una de estas configuraciones, se volverán a generar las máscaras de predicción para cada imagen y se guardarán en archivos `.npy` en una nueva subcarpeta dentro del `OUTPUT_DIR`.

In [ ]:
MASKS_SAVE_DIR = os.path.join(OUTPUT_DIR, "top5_ap_cellpose_masks")
os.makedirs(MASKS_SAVE_DIR, exist_ok=True)
print(f"Las máscaras se guardarán en: {MASKS_SAVE_DIR}\n")

# Obtener las 5 mejores configuraciones (modelo, mask_threshold, flow_threshold)
# de todas las combinaciones, ordenadas por 'ap_cellpose'.
top5_configs_all = ranking_configs.sort_values(by='ap_cellpose', ascending=False).head(5)

print("Las 5 mejores configuraciones por ap_cellpose para guardar las máscaras:")
display(top5_configs_all)

all_tif_files = get_tif_files(IMAGES_DIR)

for idx, config_row in top5_configs_all.iterrows():
    model_name = config_row["model"]
    mask_threshold = config_row["mask_threshold"]
    flow_threshold = config_row["flow_threshold"]

    print(f"\nProcesando configuración: Modelo={model_name}, Mask_Threshold={mask_threshold}, Flow_Threshold={flow_threshold}")
    model = models.CellposeModel(gpu=True, model_type=model_name)

    # Crear un subdirectorio para cada configuración para organizar las máscaras guardadas
    config_output_dir = os.path.join(MASKS_SAVE_DIR, f"{model_name}_mt{mask_threshold}_ft{flow_threshold}")
    os.makedirs(config_output_dir, exist_ok=True)

    for image_name, image_filepath in all_tif_files.items():
        img = cargar_imagen(image_filepath)
        pr_lab, _, _ = model.eval(
            img, mask_threshold=mask_threshold, flow_threshold=flow_threshold, **EVAL_PARAMS
        )
        save_path = os.path.join(config_output_dir, f"{image_name[:-4]}_pred.npy")
        np.save(save_path, pr_lab)
        # print(f"  Guardado: {save_path}") # Descomentar para ver el progreso detallado

print("\nGuardado de predicciones completado para las 5 mejores configuraciones.")

## Selección de configuración por píxel (por modelo) y comparación

Para cada modelo se elige también la mejor config a nivel **píxel** (por `px_f1`,
promedio sobre imágenes) y se la compara con la mejor por **instancia**. Pueden
diferir.


In [ ]:
def seleccionar_config_pixel_por_modelo(resultados_px, metrica="px_f1"):
    """Para cada modelo, promedia px-métricas sobre imágenes por config y elige la mejor."""
    cols_cfg = ["model", "mask_threshold", "flow_threshold"]
    cols_met = ["px_precision", "px_recall", "px_f1", "px_iou", "px_specificity", "px_accuracy"]
    agg = (
        resultados_px.groupby(cols_cfg)[cols_met]
        .mean()
        .reset_index()
    )
    mejores = (
        agg.sort_values(metrica, ascending=False)
        .groupby("model", as_index=False)
        .first()
    )
    return mejores, agg


mejores_px_por_modelo, ranking_configs_px = seleccionar_config_pixel_por_modelo(resultados_px, metrica="px_f1")
ranking_configs_px.to_csv(f"{OUTPUT_DIR}/config_ranking_pixel_por_modelo.csv", index=False)
print("Mejor configuración por modelo según px_f1 (promedio sobre imágenes):")
mejores_px_por_modelo

In [ ]:
# Comparación por modelo: config mejor-por-instancia vs mejor-por-píxel, con métricas de AMBOS niveles.
def _inst_f1_modelo(model_name, mask_thr, flow_thr, tau=TAU_REF):
    m = (
        (resultados["model"] == model_name) &
        (resultados["tau"] == tau) &
        (resultados["mask_threshold"] == mask_thr) &
        (resultados["flow_threshold"] == flow_thr)
    )
    return resultados[m]["f1"].mean()


def _px_f1_modelo(model_name, mask_thr, flow_thr):
    m = (
        (resultados_px["model"] == model_name) &
        (resultados_px["mask_threshold"] == mask_thr) &
        (resultados_px["flow_threshold"] == flow_thr)
    )
    return resultados_px[m]["px_f1"].mean()


inst_idx = mejores_por_modelo.set_index("model")
px_idx = mejores_px_por_modelo.set_index("model")
filas_cmp = []
for model_name in MODELS:
    ci, cp = inst_idx.loc[model_name], px_idx.loc[model_name]
    filas_cmp.append({
        "model": model_name, "seleccion": "instancia (f1)",
        "mask_threshold": ci["mask_threshold"], "flow_threshold": ci["flow_threshold"],
        "f1_instancia": _inst_f1_modelo(model_name, ci["mask_threshold"], ci["flow_threshold"]),
        "px_f1": _px_f1_modelo(model_name, ci["mask_threshold"], ci["flow_threshold"]),
    })
    filas_cmp.append({
        "model": model_name, "seleccion": "pixel (px_f1)",
        "mask_threshold": cp["mask_threshold"], "flow_threshold": cp["flow_threshold"],
        "f1_instancia": _inst_f1_modelo(model_name, cp["mask_threshold"], cp["flow_threshold"]),
        "px_f1": _px_f1_modelo(model_name, cp["mask_threshold"], cp["flow_threshold"]),
    })

comparacion_niveles = pd.DataFrame(filas_cmp)
comparacion_niveles.to_csv(f"{OUTPUT_DIR}/comparacion_instancia_vs_pixel.csv", index=False)
print("Comparación config instancia vs píxel (por modelo):")
comparacion_niveles

## Comparación entre modelos

Cada modelo con su mejor configuración. Ranking por f1 y por `ap_cellpose`.


In [ ]:
comparacion = mejores_por_modelo.sort_values("f1", ascending=False).reset_index(drop=True)
comparacion.to_csv(f"{OUTPUT_DIR}/comparacion_modelos.csv", index=False)
print("Comparación de modelos (mejor config de cada uno):")
print(comparacion.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comparacion))
w = 0.35
ax.bar(x - w / 2, comparacion["f1"], w, label="f1")
ax.bar(x + w / 2, comparacion["ap_cellpose"], w, label="ap_cellpose")
ax.set_xticks(x)
ax.set_xticklabels(comparacion["model"], rotation=20, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel(f"valor (promedio sobre imágenes, τ={TAU_REF})")
ax.set_title("Comparación de modelos Omnipose — mejor config de cada uno")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Curvas de métricas vs τ (mejor config de cada modelo)

In [ ]:
plt.figure(figsize=(9, 5))
for _, fila in mejores_por_modelo.iterrows():
    mask_cfg = (
        (resultados["model"] == fila["model"]) &
        (resultados["mask_threshold"] == fila["mask_threshold"]) &
        (resultados["flow_threshold"] == fila["flow_threshold"])
    )
    curva = resultados[mask_cfg].groupby("tau")["f1"].mean()
    plt.plot(curva.index, curva.values, marker="o", label=fila["model"])

plt.xlabel("τ (umbral de IoU)")
plt.ylabel("f1 (promedio sobre imágenes)")
plt.title("f1 vs τ — mejor config de cada modelo")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Visualización cualitativa de errores

Overlay sobre la imagen original: **verde = TP**, **rojo = FP**, **azul = FN**
(mismo matching húngaro que las métricas, vía `match_instances`). Una grilla 3×3
por modelo, con su mejor configuración, sobre los nueve frames de referencia.


In [ ]:
def visualizar_prediccion(img, gt_lab, pr_lab, tau=TAU_REF, title=None, ax=None):
    """Overlay: verde=TP, rojo=FP, azul=FN. Reutiliza match_instances."""
    m = match_instances(gt_lab, pr_lab, tau)
    overlay = np.zeros((*gt_lab.shape, 3), dtype=np.float32)
    for p in m["tp_pr_ids"]:
        overlay[pr_lab == p] = [0, 1, 0]
    for p in m["fp_pr_ids"]:
        overlay[pr_lab == p] = [1, 0, 0]
    for g in m["fn_gt_ids"]:
        overlay[gt_lab == g] = [0, 0, 1]

    creado = ax is None
    if creado:
        _, ax = plt.subplots(figsize=(6, 6))
    im = img.astype(float)
    if im.max() > im.min():
        im = (im - im.min()) / (im.max() - im.min())
    ax.imshow(im, cmap="gray")
    ax.imshow(overlay, alpha=0.4)
    ax.set_title(title or f"TP/FP/FN — τ={tau}")
    ax.axis("off")
    if creado:
        plt.show()


def visualizar_modelo(model_name, cfg, tau=TAU_REF):
    """Grilla 3x3 de overlays para un modelo con su config, en los frames de ref."""
    model = models.CellposeModel(gpu=True, model_type=model_name)
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))
    for ax, (label, stem) in zip(axes.ravel(), FRAMES_REFERENCIA.items()):
        image_filepath = f"{IMAGES_DIR}/{stem}.tif"
        gt_lab = get_ground_truth(image_filepath)["masks"]
        img = cargar_imagen(image_filepath)
        pr_lab, _, _ = model.eval(
            img,
            mask_threshold=cfg["mask_threshold"],
            flow_threshold=cfg["flow_threshold"],
            **EVAL_PARAMS,
        )
        visualizar_prediccion(img, gt_lab, pr_lab, tau=tau, title=label, ax=ax)
    plt.suptitle(f"{model_name} — mask_thr={cfg['mask_threshold']}, "
                 f"flow_thr={cfg['flow_threshold']} — τ={tau}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Renderizar la grilla de errores para cada modelo con su mejor config
for _, fila in mejores_por_modelo.iterrows():
    cfg = {
        "mask_threshold": fila["mask_threshold"],
        "flow_threshold": fila["flow_threshold"],
    }
    visualizar_modelo(fila["model"], cfg, tau=TAU_REF)

## Superposición pixel a pixel (estilo Ilastik)

Mapa por píxel: **negro = fondo**, **blanco = TP**, **verde = FN**, **rojo = FP**.
Por cada modelo se grafica su mejor config por **instancia** y por **píxel**, para
compararlas.


In [ ]:
def grilla_superposicion_pixel(model_name, cfg, titulo):
    """Grilla 3x3 de superposición pixel a pixel para un modelo con su config."""
    model = models.CellposeModel(gpu=True, model_type=model_name)
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))
    for ax, (label, stem) in zip(axes.ravel(), FRAMES_REFERENCIA.items()):
        image_filepath = f"{IMAGES_DIR}/{stem}.tif"
        gt_lab = get_ground_truth(image_filepath)["masks"]
        img = cargar_imagen(image_filepath)
        pr_lab, _, _ = model.eval(
            img,
            mask_threshold=cfg["mask_threshold"],
            flow_threshold=cfg["flow_threshold"],
            **EVAL_PARAMS,
        )
        sup = pixel_superposition(gt_lab, pr_lab)
        ax.imshow(sup, cmap=_PX_CMAP, vmin=0, vmax=3, interpolation="nearest")
        ax.set_title(label)
        ax.axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["black", "white", "green", "red"]]
    fig.legend(handles, ["Fondo", "TP", "FN", "FP"], loc="lower center", ncol=4, fontsize=12)
    plt.suptitle(titulo, fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.06)
    plt.show()


for model_name in MODELS:
    ci = inst_idx.loc[model_name]
    cp = px_idx.loc[model_name]
    grilla_superposicion_pixel(
        model_name,
        {"mask_threshold": ci["mask_threshold"], "flow_threshold": ci["flow_threshold"]},
        f"{model_name} — superposición píxel (config instancia)",
    )
    grilla_superposicion_pixel(
        model_name,
        {"mask_threshold": cp["mask_threshold"], "flow_threshold": cp["flow_threshold"]},
        f"{model_name} — superposición píxel (config píxel)",
    )

---
## Apéndice — Instalación local con conda (receta oficial de la guía)

La celda de instalación de arriba es la adaptación a **Colab**. Si en cambio corrés
Omnipose **localmente**, la guía recomienda un entorno conda con **Python 3.10**
(hay bugs en 3.11+):

```bash
# CPU:
conda create -n omnipose 'python==3.10.12' pytorch
# GPU (NVIDIA, CUDA 11.8): agregar
#   torchvision pytorch-cuda=11.8 -c pytorch -c nvidia
conda activate omnipose
pip install omnipose

# Si ya tenías Omnipose instalado, primero limpiá para evitar conflictos:
#   pip uninstall cellpose_omni && pip cache remove cellpose_omni
```

Versiones de Python confirmadas: 3.8.5 – 3.10.11. En Colab no se reproduce este
entorno tal cual (condacolab rompe el montaje de Drive), por eso allá usamos pip
con `ncolor<2`.
